In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib
import json
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow.keras.models import load_model

warnings.filterwarnings('ignore')
sns.set()
sns.set_context('talk')

In [ ]:
# ── Data ────────────────────────────────────────────────────────────────────
with open('../../config.json') as f:
    cfg = json.load(f)

target    = cfg['rf']
file_name = cfg['rf_file_name']

data = pd.read_csv(f'../../data/{file_name}.csv')
X = data.drop(columns=[target])
y = data[target]

# Seeds utilisées lors de l'entraînement (même générateur que dans les notebooks)
master_rng = np.random.default_rng(seed=99)
seeds = master_rng.choice(10000, size=4, replace=False).tolist()
print("Seeds :", seeds)

In [ ]:
# ── Évaluation par seed puis moyenne ────────────────────────────────────────
import os
from tensorflow.keras.models import load_model

model_types = ['Random Forest', 'XGBoost', 'ANN']
per_seed_results = {mt: [] for mt in model_types}

model_files = {
    'Random Forest': lambda s: f'saved_models/random_forest_seed_{s}.pkl',
    'XGBoost':       lambda s: f'saved_models/xgboost_seed_{s}.pkl',
    'ANN':           lambda s: f'saved_models/ann_seed_{s}.keras',
}

for seed in seeds:
    X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
        X, y, test_size=0.2, random_state=seed
    )

    scaler_path = f'saved_models/scaler_seed_{seed}.pkl'
    scaler_s = joblib.load(scaler_path) if os.path.exists(scaler_path) else None
    X_test_scaled_s = (
        pd.DataFrame(scaler_s.transform(X_test_s), columns=X_test_s.columns)
        if scaler_s else None
    )

    for name in model_types:
        path = model_files[name](seed)
        if not os.path.exists(path):
            print(f"[SKIP] {path} introuvable")
            continue
        if name == 'ANN' and X_test_scaled_s is None:
            print(f"[SKIP] ANN seed={seed} — scaler manquant")
            continue

        model = load_model(path) if name == 'ANN' else joblib.load(path)
        y_pred = model.predict(X_test_scaled_s, verbose=0).flatten() if name == 'ANN' else model.predict(X_test_s)

        per_seed_results[name].append({
            'R²':   r2_score(y_test_s, y_pred),
            'MAE':  mean_absolute_error(y_test_s, y_pred),
            'RMSE': np.sqrt(mean_squared_error(y_test_s, y_pred)),
        })

# ── Moyenne et écart-type sur les seeds disponibles ─────────────────────────
rows = []
for name in model_types:
    results = per_seed_results[name]
    if not results:
        print(f"[WARN] Aucun modèle disponible pour {name}")
        continue
    r2s   = [d['R²']   for d in results]
    maes  = [d['MAE']  for d in results]
    rmses = [d['RMSE'] for d in results]
    rows.append({
        'Model':     name,
        'N seeds':   len(results),
        'R² mean':   round(np.mean(r2s),   4),
        'R² std':    round(np.std(r2s),    4),
        'MAE mean':  round(np.mean(maes),  4),
        'MAE std':   round(np.std(maes),   4),
        'RMSE mean': round(np.mean(rmses), 4),
        'RMSE std':  round(np.std(rmses),  4),
    })

metrics_df = pd.DataFrame(rows).set_index('Model')
print(metrics_df.to_string())

In [ ]:
# ── 1. Metrics bar chart (mean ± std) ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
colors = ['#4C72B0', '#DD8452', '#55A868']

for ax, metric in zip(axes, ['R²', 'MAE', 'RMSE']):
    means = metrics_df[f'{metric} mean']
    stds  = metrics_df[f'{metric} std']
    bars  = ax.bar(metrics_df.index, means, yerr=stds, color=colors,
                   capsize=6, error_kw={'elinewidth': 1.5})
    ax.set_title(metric)
    ax.set_ylabel(f'{metric} (mean ± std)')
    ax.set_xticklabels(metrics_df.index, rotation=15, ha='right')
    for bar, val, std in zip(bars, means, stds):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + std + 0.005 * bar.get_height(),
                f'{val:.3f}', ha='center', va='bottom', fontsize=10)

plt.suptitle('Model Comparison — RF  [mean over 4 seeds]', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('saved_models/comparison_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 2. Actual vs Predicted — seed représentative (première seed disponible) ───
available_models = [
    name for name in model_types
    if per_seed_results[name]  # au moins une seed entraînée
]

# Trouve la première seed pour laquelle tous les modèles disponibles existent
seed_ref = next(
    s for s in seeds
    if all(os.path.exists(model_files[n](s)) for n in available_models)
)

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y, test_size=0.2, random_state=seed_ref)
scaler_path_r = f'saved_models/scaler_seed_{seed_ref}.pkl'
scaler_r    = joblib.load(scaler_path_r) if os.path.exists(scaler_path_r) else None
X_test_sc_r = pd.DataFrame(scaler_r.transform(X_test_r), columns=X_test_r.columns) if scaler_r else None

preds_ref = {}
for name in available_models:
    if name == 'ANN' and X_test_sc_r is None:
        continue
    model = load_model(model_files[name](seed_ref)) if name == 'ANN' else joblib.load(model_files[name](seed_ref))
    preds_ref[name] = model.predict(X_test_sc_r, verbose=0).flatten() if name == 'ANN' else model.predict(X_test_r)

colors_ref = ['#4C72B0', '#DD8452', '#55A868'][:len(preds_ref)]
fig, axes = plt.subplots(1, len(preds_ref), figsize=(6*len(preds_ref), 6))
if len(preds_ref) == 1:
    axes = [axes]

for ax, (name, y_pred), color in zip(axes, preds_ref.items(), colors_ref):
    ax.scatter(y_test_r, y_pred, alpha=0.6, color=color, edgecolors='white', linewidths=0.5)
    lims = [min(y_test_r.min(), y_pred.min()), max(y_test_r.max(), y_pred.max())]
    ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect fit')
    ax.set_xlabel('Actual RF')
    ax.set_ylabel('Predicted RF')
    r2   = r2_score(y_test_r, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test_r, y_pred))
    ax.set_title(f'{name}\nR² = {r2:.3f}  |  RMSE = {rmse:.3f}')
    ax.legend(fontsize=9)

plt.suptitle(f'Actual vs Predicted — RF  [seed={seed_ref}]', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('saved_models/comparison_actual_vs_pred.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 3. Residuals distribution (seed représentative) ──────────────────────────
fig, axes = plt.subplots(1, len(preds_ref), figsize=(6*len(preds_ref), 5))
if len(preds_ref) == 1:
    axes = [axes]

for ax, (name, y_pred), color in zip(axes, preds_ref.items(), colors_ref):
    residuals = np.array(y_test_r) - y_pred
    ax.hist(residuals, bins=20, color=color, edgecolor='white', alpha=0.8)
    ax.axvline(0, color='red', linestyle='--', linewidth=1.5)
    ax.set_xlabel('Residual (Actual − Predicted)')
    ax.set_ylabel('Count')
    ax.set_title(f'{name} — Residuals')

plt.suptitle(f'Residuals Distribution — RF  [seed={seed_ref}]', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('saved_models/comparison_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 4. Résumé final ───────────────────────────────────────────────────────────
print('===== Final Metrics Summary (mean ± std over 4 seeds) =====')
print(metrics_df.to_string())

print(f'\nBest R²:   {metrics_df["R² mean"].idxmax()} ({metrics_df["R² mean"].max():.4f} ± {metrics_df.loc[metrics_df["R² mean"].idxmax(), "R² std"]:.4f})')
print(f'Best MAE:  {metrics_df["MAE mean"].idxmin()} ({metrics_df["MAE mean"].min():.4f} ± {metrics_df.loc[metrics_df["MAE mean"].idxmin(), "MAE std"]:.4f})')
print(f'Best RMSE: {metrics_df["RMSE mean"].idxmin()} ({metrics_df["RMSE mean"].min():.4f} ± {metrics_df.loc[metrics_df["RMSE mean"].idxmin(), "RMSE std"]:.4f})')